<img src="img/swdb_logo.jpg" width="900" style="display: block; margin: 0 auto;">

<h1 align="center">Connectomics Exercise 2: Which interneurons target which compartments?</h1>
<h3 align="center">Summer Workshop on the Dynamic Brain 2026</h3>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p>A pyramidal cell is not inhibited uniformly. Different inhibitory subclasses place their synapses in different places on it, and that arrangement is a large part of how these subclasses differ. We will start by focusong on <b>L5-ET</b> cells — thick-tufted, subcortically projecting, with some of the most elaborate dendrites in cortex.
<p>We will build also everything directly from CAVE to get used to working with direct data access.
<p>CAVE stands for Connectome Annotation Versioning Engine, and it powers proofreading and analysis of the large EM datasets we work with.
<p>Disclaimer: Claude was used to help generate exercise details and solutions.

</div>

In [ ]:
import sys
from os.path import join as pjoin

mat_version = 1196
sys.path.append(pjoin("..", "utils"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import ossify as osy
from caveclient import CAVEclient
from standard_transform.datasets import v1dd_ds

from utils import cell_type_palette

sns.set_context("notebook")
plt.rcParams["figure.dpi"] = 110

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p>To run in Code Ocean, leave the setup_token line and follow the instructions.
<p>If running locally, the token will be saved after first use and you won't need to save keep running setup_token.

</div>

In [ ]:
CAVEclient.setup_token("https://global.em.brain.allentech.org")
client = CAVEclient(datastack_name="v1dd_public", version=mat_version)

<div style="border-left: 3px solid #B8860B; padding: 1px; padding-left: 10px; background: #FFF8DC; ">

<p><b>If a CAVE query takes longer than three minutes, stop it.</b>
<p>The CAVE queries in this notebook should return in seconds. Past about three minutes it is not going to finish on its own — interrupt the kernel (the ■ button, or <i>Kernel → Interrupt</i>) and run the cell again.
<p>If the same cell hangs a second time, flag a TA rather than waiting it out. It usually means the server is under load or the request is larger than it looks, and both are worth someone checking.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3> The CAVE calls you will need </h3>
<p>Every query in this notebook is one of four shapes. The point of the exercises is the analysis, not remembering the API, so here they all are.
<p><b>Discover what exists.</b>
<pre><code>
client.materialize.get_tables()          # queryable tables
client.materialize.get_views()           # views (pre-joined convenience tables)
client.materialize.get_versions()        # materialization versions
</code></pre>
<p><b>Pull a whole table.</b> The <code>tables.&lt;name&gt;()</code> accessor takes filters as keyword arguments; <code>.get_all()</code> skips the filter and returns everything.
<pre><code>
df = client.materialize.tables.cell_type_multifeature_v1().get_all(
    split_positions=True,          # pt_position -> pt_position_x / _y / _z
    desired_resolution=[1, 1, 1],  # positions in NANOMETRES rather than voxels
)

df = client.materialize.tables.proofreading_status_and_strategy(
    status_axon=True,              # any column can be a filter kwarg
).query(split_positions=True)
</code></pre>
<p><b>Query synapses.</b> <code>pre_ids</code> and <code>post_ids</code> each take a list of root ids. Giving both is a logical AND — only synapses from those cells onto those cells — which is how you ask a narrow question cheaply instead of filtering a huge answer afterwards.
<pre><code>
syn = client.materialize.synapse_query(
    post_ids=some_root_ids,                  # or pre_ids=..., or both
    desired_resolution=[1, 1, 1],
    split_positions=True,
)
</code></pre>
<p><b>Query a table by an explicit list of ids.</b> Needed for reference tables such as the synapse compartment predictions, which are keyed to synapse ids rather than to cells.
<pre><code>
client.materialize.query_table(
    "synapse_target_predictions_ssa",
    filter_in_dict={"target_id": [int(i) for i in some_synapse_ids]},
    select_columns=["target_id", "tag"],
    merge_reference=False,   # return the reference table's own rows, unjoined
    log_warning=False,
)
</code></pre>
<p>Two things that will bite you regardless of which shape you use:
<ul>
<li><b>Root ids come back as a nullable <code>Int64</code></b>, which is not JSON-serialisable. Passing
</ul>
<p><code>df.pt_root_id.values</code> into a query raises <code>TypeError: Object of type IntegerArray is not JSON serializable</code>. Use <code>root_ids()</code> below.
<ul>
<li><b>A query that is too large returns <code>502 Bad Gateway</code></b>, not a helpful error. The limit is on the size of the response, not the number of ids. Question 4 makes you meet this properly.
</ul>

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Example Cells</h3>
<p> Most Layer 5 ET cells would work, but these ones definitely will.

</div>

In [ ]:
EXEMPLAR_L5ET = 864691132734343191
BACKUP_L5ET = [864691132830244868, 864691132669405014]

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p>The helper the note above refers to:

</div>

In [ ]:
def root_ids(series):
    """Root ids as a plain uint64 array, safe to hand to a CAVE query."""
    return series.dropna().astype("uint64").to_numpy()

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p><b>Coordinates come in three flavours and mixing them fails silently but gives you wrong answers.</b>
Voxels, raw nanometres, and <i>transformed</i> microns (rotated so the pia is flat, with y as cortical depth). Since transformed values are the best to work with, it's easiest to push every transform through one helper so it cannot happen:

</div>

In [ ]:
def to_transformed_um(df, prefix):
    """Columns f'{prefix}_x/y/z' in nanometres -> transformed microns, pia-flat."""
    cols = [f"{prefix}_{ax}" for ax in "xyz"]
    return v1dd_ds.transform("nm").apply(df[cols].values)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p><b>A synapse's position relative to the cell it lands on.</b> Question 3 will ask you to compute the location of a synapse relative to the cell body in spherical coordinates. Feel free to write your own function, but rather than get hung up on geometry you can also use this function to accomplish that. Given a set of points and a soma, both in transformed µm, it returns how far each point is from that soma and in what direction — with the compass flattened to a single angle.

</div>

In [ ]:
def r_theta(points_um, soma_um):
    """Spherical-ish coordinates about a soma, with azimuth dropped.

    r     : straight-line distance from the soma, µm
    theta : degrees out of the tangential plane; +90 = straight toward the pia,
            0 = level with the soma, -90 = straight down.

    Dropping azimuth assumes the target is roughly rotationally symmetric about the
    pia axis -- fair for a pyramidal cell, wrong for anything with a planar arbor.
    """
    delta = points_um - soma_um
    lateral = np.hypot(delta[:, 0], delta[:, 2])
    return (
        np.linalg.norm(delta, axis=1),
        np.degrees(np.arctan2(-delta[:, 1], lateral)),
    )

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p><b>Plotting a synapse cloud.</b> The function <code>cloud</code> draws synapses relative to the target cell body. This will be used in Question 4.

</div>

In [ ]:
def cloud(ax, df, title, color):
    """Every synapse in the frame of the cell it landed on. Azimuth was collapsed, so
    lateral distance is a magnitude -- mirror it, which is what the rotational-symmetry
    assumption already claims."""
    lat = df.r_um * np.cos(np.radians(df.theta_deg))
    vert = df.r_um * np.sin(np.radians(df.theta_deg))
    ax.scatter(
        np.r_[lat, -lat], np.r_[vert, vert], s=3.0, alpha=0.18, lw=0, color=color
    )
    ax.scatter([0], [0], marker="*", s=180, c="k", zorder=6)
    for rad in (25, 50, 100, 150):
        ax.add_artist(plt.Circle((0, 0), rad, fill=False, color="0.6", lw=0.7, ls=":"))
    ax.set(
        xlim=(-160, 160),
        ylim=(-160, 200),
        aspect="equal",
        title=title,
        xlabel="lateral distance (µm)",
    )
    # The frame is clipped so the tight cloud is readable; say how much falls outside
    # rather than letting the crop quietly imply the cloud ends there.
    out = 100 * (df.r_um > 160).mean()
    if out > 1:
        ax.text(
            0.97,
            0.03,
            f"{out:.0f}% beyond frame",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=8,
            color="0.4",
        )

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<hr>
<h2> Question 1 — Who are the L5-ET cells, and what do you need to select to work with reconstructions that you trust? </h2>
<p><b>ET</b> is <i>extratelencephalic</i>: They are a kind of pyramidal cell that projects out of cortex, to thalamus, superior colliculus and/or brainstem. They are also known as thick-tufted cells, with a long apical trunk rising to a (sometimes very wide) tuft in Layer 1, plus a basal dendrite around the soma.
<p>Inhibitory cells come classified into four subclasses — <code>PTC</code>, <code>DTC</code>, <code>ITC</code>, <code>STC</code> — by <code>cell_type_multifeature_v1</code>, a classifier built on <b>soma, nucleus, dendrite and spine morphology</b>. Note what is <i>not</i> in that list: the classifier never sees where a cell's synapses land.
<p><b>Treat the four labels as opaque for now.</b> They are four groups that look different down a microscope, nothing more. Whether they also <i>behave</i> differently — whether they put their synapses in different places on a target — is the question this module aims to answer. The names do stand for something, and we come back to them at the end; unpacking them now would hand you the answer before you have measured anything.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 1:</b> Build a table of all classified cells from <code>cell_type_multifeature_v1</code> and remove any duplicate root ids that do not have a well-defined type. How many of each type, and which are excitatory vs inhibitory?
<p>Note: The table query function is <code>client.materialize.query_table</code>, or you can also use <code>client.materialize.tables.&lt;table&gt;.query</code>
<p><i>Solution note: Later cells expect this table to be called <code>cell_df</code>.</i>

</div>

In [ ]:
# Your code here

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p>Note the spelling: <b><code>L5ET</code>, without a hyphen.</b> Other modules in this workshop load derived data products that spell the same type <code>L5-ET</code>.
<p>The shared palette in <code>utils</code> is keyed the hyphenated way too, so bridge the two spellings once rather than maintaining a second dict.

</div>

In [ ]:
import re  # Regex is a tool to search for and replace strings based on matching various patterns.


def hyphenated(t):
    """CAVE spells excitatory types 'L5ET'; cell_type_palette keys them 'L5-ET'."""
    return re.sub(r"^(L\d)([A-Z]+)$", r"\1-\2", str(t))


PALETTE = {
    t: cell_type_palette.get(hyphenated(t), "0.6")
    for t in cell_df.cell_type.dropna().unique()
}
print({k: PALETTE[k] for k in list(PALETTE)[:4]})

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p>Soma depth by cell type is worth plotting before going further — it is the sanity check that our coordinate transform is working as expected.

</div>

In [ ]:
# Your code here

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 2:</b> Load <code>EXEMPLAR_L5ET</code> and plot it. Can you tell from the picture why it is called a thick-tufted cell>

</div>

In [ ]:
# Your code here

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p>The apical trunk runs from the soma at roughly 470 µm up to a tuft that reaches the pia, and there is a basal skirt at the soma. That is the thick-tufted ET morphology.
<p><b>The axon is in there too</b>, sprawling laterally. It belongs to the <i>presynaptic</i> side of this cell's story, not the postsynaptic side we care about, and Question 3 has to remove it before measuring anything along the dendrite.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3> Whose reconstruction has to be right? </h3>
Not every cell in the dataset has the same quality. Here, we're going to identify which cells are proofread so we can work with them as needed.
</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 3:</b> Build the presynaptic universe of cell typed and proofread cells. Join <code>proofreading_status_and_strategy</code> to the cell type table and count, per inhibitory subclass, how many cells have <code>status_axon == True</code>. Then look at <code>strategy_axon</code> and ask what the partially-extended cells are.
<p><i>Call the joined, filtered frame <code>inh_pf</code>.</i>

</div>

In [ ]:
# Your code here

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Questions — no code.</b>
<p>We are about to study synapses <i>onto</i> an L5-ET cell.
<ol>
<li><b>Whose reconstruction quality matters, the ET cell's or the interneurons'?</b>
<li><b>A trusted dendrite means every synapse was found. Does it mean every synapse was correctly attributed?</b>
<li><b>Reconstruction loses thin distal branches more often than thick proximal ones. Suppose one subclass tends to synapse close to its targets' somata and another travels far out along the dendrite. Without the filter, which of the two would be over-represented?</b>
</ol>
<p><code>strategy_axon</code> further splits proofread cells into fully and partially extended, which is <b>completeness, not correctness</b> — a partially extended axon has had its merges cleaned, so its synapses are still correctly attributed. Completeness matters when you need a per-axon denominator, but here are okay with partial (but substantial) reconstructions.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<hr>
<h2> Question 2 — What lands where? </h2>
<p>Synapses are labelled by what they land on: a dendritic <b>spine</b>, a dendritic <b>shaft</b>, or the <b>soma</b>. A synapse on the soma sits right where the axon initial segment decides whether to fire, so it can shunt the whole cell. The textbook claims are:
<ol>
<li>Excitatory → excitatory synapses are onto <b>spines</b>.
<li>Inhibitory → excitatory synapses are onto <b>shafts</b>.
<li>Synapses onto inhibitory cells are onto <b>shafts</b> because inhibitory dendrites are much less spiny (and might not even have well defined spines).
<li>Basket cells target <b>cell bodies</b> — so one of the four inhibitory labels should own most somatic synapses.
</ol>

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 1 — assemble the synapse table.</b> We want a dataframe with one row per synapse onto the exemplar L5ET, with a compartment and a presynaptic subclass label the input synapses. Grab the input synapses, their the compartment labels, and the presynaptic cell type and restrict to proofread axons. Report the synapse count before and after that last step.
<p>If you want to get better statistics, you can also make a group of more than one L5-ET cell — for example, take the closest 10-15 ET cells in space and grab all their input synapses.
</div>

In [ ]:
# Your code here

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p>The compartment label lives in <code>synapse_target_predictions_ssa</code>, a reference table keyed on synapse id.
<p><code>fetch_tags</code> splits the request into chunks, since the server will not take it all at once.

</div>

In [ ]:
def fetch_tags(syn_ids, chunk=40_000):
    """Compartment label (spine / shaft / soma) for each synapse id."""
    out = []
    for i in range(0, len(syn_ids), chunk):
        out.append(
            client.materialize.query_table(
                "synapse_target_predictions_ssa",
                filter_in_dict={"target_id": list(map(int, syn_ids[i : i + chunk]))},
                select_columns=["target_id", "tag"],
                merge_reference=False,
                log_warning=False,
            )
        )
    return pd.concat(out, ignore_index=True).rename(columns={"target_id": "id"})

In [ ]:
# Your code here

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p>Now restrict to cells with a proofread axon and a cell type label. This restricts the analysis set quite a lot, but still leaves a lot of measured inputs.

</div>

In [ ]:
# Your code here

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 2:</b> Build the contingency table of presynaptic type × target compartment, and normalize it <b>by row</b>. This is a table where each row will be a cell type, each column will be a compartment, and the value will be the number of synapses in that compartment. You can then ask why the column version — "who owns this compartment?" — is not a question you can answer here.

</div>

In [ ]:
# Your code here

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p><b>Read this table across the rows!</b>
<p>A row asks <b>where a given type puts its synapses.</b> It is conditional on the type, so it does not care how well any <i>other</i> type happens to be sampled — which is why the excitatory row and the inhibitory rows can come from different queries and still both be sound. Each row is its own denominator.
<p>What aspects of this hold with the textbook and which do not?

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p>Before doing more statistics, let's make one picture. Here are the exemplar's input synapses on its own morphology, coloured by presynaptic subclass — one cell, not the result.

</div>

In [ ]:
# Your code here

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<hr>
<h2> Question 3 — Where along the dendrite are these synapses? </h2>
<p>"Shaft" covers a synapse 20 µm from the soma and one 400 µm out on the apical tuft. Electrically these have very different meaning, so a compartment label is not enough — we need a position along the arbor.
<p>Perhaps the best measure would <b>geodesic distance</b>: path length through the dendrite from the soma. It is what sets how much of a synapse's effect survives the trip, and it needs a <b>skeleton</b>, which is why this question works on the one screened cell where we have one.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 1 — get the distances.</b> Compute geodesic distance to the soma for every input synapse on <code>EXEMPLAR_L5ET</code>. This is just like what we did in Module 1.

</div>

In [ ]:
# Your code here

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 2 — the distribution.</b> Plot a histogram of geodesic distance for the proofread inhibitory inputs, split by the four subclasses. Do they sit in different places along the dendrite?

</div>

In [ ]:
# Your code here

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 3, summary statistics.</b> Compute the median and quartiles of geodesic distance per subclass.

</div>

In [ ]:
# Your code here

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p>Two cautions: <b><code>ITC</code> have very few synapses onto these cells</b>, so their median is quite noisy; the next question fixes that by pooling over many targets. Also, the <b>a raw histogram is partly a picture of where the dendrite is</b> — more synapses at 100 µm may only mean more dendrite at 100 µm. That confound does not disturb the <i>ranking</i> here, because all subclasses are measured against the same dendrite.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<hr>
<h2>Question 4 — How does this generalize across pre- and posts-synaptic cells? </h2>
<p>Question 3 required a skeleton, but we don't want to compute hundres or thousands of skeletons right now. For the next question, geodesic distance is out of reach.
<p>One approximation is to look at the target's <b>soma position</b>, which is just a column — <code>r_theta</code> from the setup. But it is an approximation, so let's first measure how good it is on the one cell where both measures exist.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 1 — how good is straight-line distance?</b> Compute <b>r</b> for the same synapses and plot a scatterplot against the geodesic distance you already have.

</div>

In [ ]:
# soma_um = v1dd_ds.transform("nm").apply(root_xyz.reshape(1, 3))[0]
# syn_um  = v1dd_ds.transform("nm").apply(anno.vertices)
# syn_df["r_um"], syn_df["theta_deg"] = r_theta(syn_um, soma_um)
# ...then colour the skeleton by theta and look at it.

# Your code here

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 2 — does the ranking survive?</b> Recompute the subclass comparison with <b>r</b> and <b>θ</b> alongside the geodesic. An approximation that blurs the axis but preserves the ordering is fine for comparing populations.

</div>

In [ ]:
# Your code here

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p><b>Choosing the exemplars.</b> Filter on <code>status_axon</code> for correctness as always.
</div>

In [ ]:
# Your code here

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 3 — start with one of each.</b> Pull the output synapses of one <code>PTC</code> and one <code>DTC</code>, and place every synapse in the frame of <i>the target it landed on</i>. Plot the two side by side. How do they differ?
</div>

In [ ]:
# Your code here

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 4 — increase the number of presynaptic interneurons to all proofread cells.</b> If we do it the obvious way, the query will probably fail. You don't need to actually run this unless you want.
</div>

In [ ]:
# Your code here

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p><b>If you just make one big query, it will almost certainly fail!</b>
<p><b>It is not a bug in your code and not bad data.</b> The traceback ends in <code>requests.HTTPError</code>, not a <code>KeyError</code> — your request was well-formed and the <i>server</i> refused it. A 502 with HTML in the body is a proxy giving up on a response that got too large. Recognising which layer failed is useful to debug issues.
<p><b>Here, it turns out that the limit is on response size, not id count.</b> Twenty of these cells return fine; forty probably do not. Twenty cells with bigger axons might also not. So "find a batch size that works" is fragile, and it's easier to iterate over smaller chunks.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 5 — fix it.</b> Batch the request and concatenate. Ten ids per call sits inside the limit and costs a tenth of the round trips; since <code>synapse_query</code> returns <code>pre_pt_root_id</code> on every row, a <code>groupby</code> recovers per-cell results afterwards. The design choice worth making deliberately is what to do when a batch <i>does</i> fail — discarding all ten is wasteful when usually nine are fine.

</div>

In [ ]:
import time


def collect(pre_ids, chunk=10, pause=3.0):
    """Query in batches, falling back to one-at-a-time when a batch fails.

    Ten ids per request sits inside the limit and costs a tenth of the round trips.
    If a batch does fail, retrying it cell-by-cell isolates the bad one instead of
    discarding all ten. Returns (frame, ids that never came back).
    """
    frames, failed = [], []
    for i in range(0, len(pre_ids), chunk):
        batch = [int(p) for p in pre_ids[i : i + chunk]]
        try:
            frames.append(outputs_in_target_frame(batch))
        except Exception:
            time.sleep(pause)
            for pid in batch:
                try:
                    frames.append(outputs_in_target_frame([pid]))
                except Exception:
                    failed.append(pid)
    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    return out, failed

# Your code here using collect

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<p>We did two fancy bits here that one might not think to do initially. First, we <b>retry with a brief pause</b>, because a 502 can also be transient and a loop that dies on iteration 17 has wasted time. And we also <b>collect failures rather than raising</b>, so one bad cell costs you one cell rather than the whole run and you can rerun the missing part.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 6 — do the two example cells hold up?</b> Redraw the same clouds from the full panel.

</div>

In [ ]:
# Your code here

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 7 — repeatability.</b> Pooling over targets means we don't know if presynaptic cells behave the same for all targets. So let's split by target: for each (interneuron, target) pair compute a median r, and look at the spread <i>within</i> one interneuron. Does a single cell do the same thing to all of its targets, or is it proximal on some and distal on others?
<p>Make a dataframe where each row is a proofread interneuron (with known type) and there are columns for the median radial distance to soma as well as the interquartile interval (25-to-75th percentile range) of the radial distance distribution. Plot 1) a histogram of the median distance distribution by interneuron subclass and 2) a plot showing median and IQR values for each interneuron as a dot with errorbars.

</div>

In [ ]:
# Your code here

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

<p><b>Task 8 — split by target cell type.</b> Does an interneuron treat every kind of target the same? Measure the median radial distance as a function of <i>both</i> presynaptic interneuron subclass and postsynaptic target class. For PTC and DTC cells, how does their medial targeting distance vary with target?

</div>

In [ ]:
# Your code here

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2> Summary </h2>
<ol>
<li><b>Excitatory input arrives on spines across the whole dendrite; inhibitory input arrives on shaft and soma</b> — and the soma is essentially inhibitory-only real estate.
<li><b>Which inhibitory subclass a synapse comes from predicts where it lands.</b> PTC within a few tens of microns of the soma, DTC several times further out and leaning apical.
<li><b>That is a property of the presynaptic cell.</b> One PTC does the same thing to hundreds of different targets.
</ol>
<p>And three habits worth more than the result: <b>filter for the failure mode your question actually has</b>; <b>give every count a denominator</b> (Figure 11); <b>calibrate your shortcuts</b> (Figure 7 is why Figure 8 is allowed to exist).
<h3> What the names meant </h3>
<p>Now they can be unpacked. <code>PTC</code> is <b>perisomatic targeting</b>, <code>DTC</code> <b>dendrite targeting</b>, <code>ITC</code> <b>inhibitory targeting</b>, <code>STC</code> <b>sparse targeting</b> — the nomenclature of Schneider-Mizell et al. The classifier that assigned them never saw a synapse; it worked from soma, nucleus, dendrite and spine morphology alone. So what you measured is that morphology predicts targeting well enough that the names are earned.
<p>Two things that buys you. First, it is a real prediction rather than a definition — the labels could have failed to separate, and you would have seen that. Second, it puts a number on Module 1's caveat that these labels "might align more with molecular class than exact targeting for a given neuron, particularly among ITC/VIP cells." Your <code>ITC</code> count — nine synapses on the exemplar — is exactly where that caveat bites.
<h3> Conceptual Exercises </h3>
<ol>
<li>Chandelier cells target the axon initial segment. The compartment table has no AIS class — so "what this dataset cannot tell you" is a legitimate finding. What would you need? What would the r, theta plot look like for a chandelier cell?
<li>Does synapse <code>size</code> vary with compartment, or with presynaptic subclass?
<li>Normalize r by target soma radius (<code>cell_df.volume</code>) and redo Figure 10. Does the L5ET/L4IT difference survive?
<li>Pick your own interneuron, unscreened, and run it through <code>outputs_in_target_frame</code>. Does anything look wrong? How would you know?
</ol>

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<hr>
<h3> Optional — the denominator problem </h3>
<p><i>Not needed for anything above, but useful for thinking about the data.</i>
<p>Every distance histogram in this module is partly a picture of <b>where the dendrite is</b>. More synapses at 100 µm may just mean more dendrite at 100 µm. It did not distort the comparisons here, because two subclasses measured on the same cells share the same denominator and it largely cancels — but the moment you compare <i>across</i> cells, or ask an absolute question like "how dense is inhibition on the tuft", it stops cancelling.
<p>The fix on a cell with a skeleton is to divide by <b>µm of dendrite per bin</b>. Run the cell below and compare the middle panel to the right one: the raw counts are dominated by the shape of the denominator.

</div>

In [ ]:
bins = np.arange(0, 601, 25)
inh = syn_df[syn_df.is_pf_inh & syn_df.pre_type.isin(inh_types)]
DASH = {"PTC": "-", "DTC": "-", "ITC": "--", "STC": ":"}

vert_dist = dend.skeleton.distance_to_root() / 1_000.0
vert_len = dend.skeleton.half_edge_length / 1_000.0
dendrite_um, _ = np.histogram(vert_dist, bins=bins, weights=vert_len)
centers = 0.5 * (bins[:-1] + bins[1:])

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].bar(centers, dendrite_um, width=22, color="0.8")
axes[0].set(
    xlabel="geodesic distance (µm)", ylabel="µm of dendrite", title="the denominator"
)
for t in inh_types:
    g = inh[inh.pre_type == t].geodesic_um
    if len(g) < 5:
        continue
    n, _ = np.histogram(g, bins=bins)
    axes[1].step(
        centers, n, where="mid", lw=2, color=PALETTE.get(t, "k"), ls=DASH[t], label=t
    )
    with np.errstate(divide="ignore", invalid="ignore"):
        dens = np.where(dendrite_um > 20, n / dendrite_um, np.nan)
    axes[2].step(
        centers, dens, where="mid", lw=2, color=PALETTE.get(t, "k"), ls=DASH[t], label=t
    )
axes[1].set(xlabel="geodesic distance (µm)", ylabel="synapses", title="raw count")
axes[2].set_yscale("log")
axes[2].set(
    xlabel="geodesic distance (µm)",
    ylabel="synapses per µm of dendrite",
    title="density — the same data, divided (log)",
)
axes[2].legend(frameon=False, fontsize=9)
fig.suptitle(
    "Optional figure: every count needs a denominator",
    y=1.02,
    x=0.02,
    ha="left",
    fontsize=11,
)
sns.despine()
plt.tight_layout()
plt.show()

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<ol>
<li>The denominator here is <i>this cell's</i> dendrite. To compare across cells you need a denominator that travels — excitatory synapse density is one candidate, since spiny dendrite carries them at a roughly steady rate, and it needs no skeleton at all.
<li>Everything above treats "distance from soma" as the axis. Electrotonically what matters is path length <i>and</i> diameter. <code>cell.skeleton</code> carries radius; a cable-weighted distance would be a genuinely better axis and nobody has run it on this dataset.
</ol>

</div>